> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads the raw adulterated-milk spectra from `src of adulterated milk/` and the unaligned `raman_consensus_spectra.xlsx`; the processed adulterated dataset is provided (aligned) as `data/raman_functional_intensity_matrix_bspline_n4_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

SRC_DIR = 'src of adulterated milk'

ADULTERANT_COLOR = {
    'Doxy':   '#1f77b4',
    'Doxy-R': '#aec7e8',
    'Enro':   '#ff7f0e',
    'Urea':   '#2ca02c',
    'Water':  '#9467bd',
}
CONC_MARKER = {'0.5x': 'o', '1x': 's', '2x': '^', '3x': 'D'}

## 1. File Inventory & Metadata Parsing

In [ ]:
def parse_filename(fname):
    """Extract acquisition metadata from filename.
    
    Pattern: {Adulterant} {Conc} {Laser} {Power} {Grating} {Objective}
             {Exposure} {Accumulations} {Range} {Hole}
    Example: 'Doxy 0.5x 532nm 25% 600g 50xL 8s 10times 400-3500cm-1 200hole'
    """
    stem = fname.replace('.txt', '')
    tokens = stem.split()

    # Adulterant: first token(s) — handle 'Doxy-R' which is one hyphenated word
    adulterant = tokens[0]

    # Concentration: first token matching Nx or 0.Nx
    conc = next((t for t in tokens if re.match(r'^\d+\.?\d*x$', t)), 'unknown')

    # Laser wavelength
    laser = next((t for t in tokens if re.match(r'^\d+nm$', t)), 'unknown')

    # Laser power (%)
    power = next((t for t in tokens if t.endswith('%')), 'unknown')

    # Exposure time
    exposure = next((t for t in tokens if re.match(r'^\d+s$', t)), 'unknown')

    # Accumulations
    accum = next((t for t in tokens if t.endswith('times')), 'unknown')

    # Hole size
    hole = next((t for t in tokens if t.endswith('hole')), 'unknown')

    return {
        'Adulterant':     adulterant,
        'Concentration':  conc,
        'Laser':          laser,
        'Power':          power,
        'Exposure':       exposure,
        'Accumulations':  accum,
        'Hole':           hole,
        'Filename':       fname,
    }

all_files = sorted(glob.glob(os.path.join(SRC_DIR, '*.txt')))
meta_rows = [parse_filename(os.path.basename(f)) for f in all_files]
meta_df   = pd.DataFrame(meta_rows)

print(f'Files found: {len(all_files)}')
print()
print(meta_df[['Adulterant','Concentration','Laser','Power','Exposure','Accumulations','Hole']].to_string(index=False))

## 2. Coverage Summary

In [ ]:
print('Adulterant × Concentration coverage:')
coverage = meta_df.pivot_table(index='Adulterant', columns='Concentration',
                               values='Filename', aggfunc='count', fill_value=0)
print(coverage.to_string())
print()
print('Acquisition conditions per adulterant:')
print(meta_df.groupby('Adulterant')[['Power','Exposure','Hole']].first().to_string())

## 3. Load Spectra — Flag Corrupt Files

In [ ]:
def load_spectrum(fpath):
    """Return (wavenumbers, intensities) from a LabRAM .txt file.
    Returns (None, None, error_msg) if the file is unreadable.
    """
    try:
        with open(fpath, 'rb') as fh:
            raw = fh.read()
        text  = raw.decode('latin-1')
        lines = [l for l in text.splitlines()
                 if not l.startswith('#') and l.strip() and '\t' in l]
        if len(lines) == 0:
            return None, None, 'No data lines found'
        pairs = []
        for l in lines:
            parts = l.split('\t')
            pairs.append((float(parts[0]), float(parts[1])))
        wn  = np.array([p[0] for p in pairs])
        y   = np.array([p[1] for p in pairs])
        return wn, y, None
    except Exception as e:
        return None, None, str(e)

spectra_raw  = {}   # label -> intensity array
wn_ref       = None
load_log     = []   # per-file load report
corrupt_files = []

for fpath, row in zip(all_files, meta_rows):
    label = f"{row['Adulterant']} {row['Concentration']}"
    wn, y, err = load_spectrum(fpath)

    if err is not None:
        status = f'CORRUPT — {err[:80]}'
        corrupt_files.append(os.path.basename(fpath))
    else:
        if wn_ref is None:
            wn_ref = wn
        if len(wn) != len(wn_ref):
            status = f'GRID MISMATCH — {len(wn)} pts vs expected {len(wn_ref)}'
            corrupt_files.append(os.path.basename(fpath))
        else:
            spectra_raw[label] = y
            status = f'OK  — {len(wn)} pts, wn {wn[0]:.1f}–{wn[-1]:.1f} cm⁻¹'

    load_log.append({'File': os.path.basename(fpath), 'Label': label, 'Status': status})

load_report = pd.DataFrame(load_log)
print(load_report[['Label','Status']].to_string(index=False))
print(f'\nSuccessfully loaded : {len(spectra_raw)}')
print(f'Corrupt / excluded  : {len(corrupt_files)}')
if corrupt_files:
    for f in corrupt_files:
        print(f'  ✗  {f}')

## 4. Build Raw Dataset

In [ ]:
dataset_raw = pd.DataFrame(spectra_raw, index=wn_ref)
dataset_raw.index.name = 'Raman Shift'

print(f'Raw dataset shape : {dataset_raw.shape[0]} wavenumbers × {dataset_raw.shape[1]} spectra')
print(f'Wavenumber range  : {wn_ref.min():.1f} – {wn_ref.max():.1f} cm⁻¹')
print(f'Missing values    : {dataset_raw.isna().sum().sum()}')
print()
print('Columns in dataset:')
for col in dataset_raw.columns:
    print(f'  {col}')

## 5. Duplicate Detection

Three complementary checks are applied:

| Check | Method | Criterion |
|-------|--------|----------|
| **Exact duplicates** | Column-wise equality | All intensity values identical |
| **Near-duplicates (correlation)** | Pearson r between every pair | r > 0.99999 |
| **Near-duplicates (L2 distance)** | L2 distance between every pair | L2 < 1e-6 |

In [ ]:
cols   = list(dataset_raw.columns)
F      = dataset_raw.values   # (2118, n_spectra)
n_spec = len(cols)

# ── Check 1: Exact duplicates ────────────────────────────────────────────────
exact_dup_pairs = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        if np.array_equal(F[:, i], F[:, j]):
            exact_dup_pairs.append((cols[i], cols[j]))

print(f'Exact duplicate pairs found: {len(exact_dup_pairs)}')
for a, b in exact_dup_pairs:
    print(f'  {a}  ==  {b}')

# ── Check 2: Near-duplicates by Pearson correlation ──────────────────────────
CORR_THRESHOLD = 0.99999
corr_matrix = np.corrcoef(F.T)   # (n_spec, n_spec)

near_dup_corr = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        r = corr_matrix[i, j]
        if r > CORR_THRESHOLD:
            near_dup_corr.append((cols[i], cols[j], round(r, 8)))

print(f'\nNear-duplicate pairs (r > {CORR_THRESHOLD}): {len(near_dup_corr)}')
for a, b, r in near_dup_corr:
    print(f'  {a}  ↔  {b}   r = {r}')

# ── Check 3: Near-duplicates by L2 distance ──────────────────────────────────
L2_THRESHOLD = 1e-6
near_dup_l2 = []
for i in range(n_spec):
    for j in range(i + 1, n_spec):
        d = np.linalg.norm(F[:, i] - F[:, j])
        if d < L2_THRESHOLD:
            near_dup_l2.append((cols[i], cols[j], d))

print(f'\nNear-duplicate pairs (L2 < {L2_THRESHOLD}): {len(near_dup_l2)}')
for a, b, d in near_dup_l2:
    print(f'  {a}  ↔  {b}   L2 = {d:.2e}')

## 7. Remove Duplicates & Build Final Dataset

In [ ]:
# Collect all columns flagged for removal (keep the first occurrence in each duplicate pair)
to_drop = set()

for a, b in exact_dup_pairs:
    to_drop.add(b)   # drop second occurrence
    print(f'Exact duplicate — dropping: {b}  (kept: {a})')

for a, b, r in near_dup_corr:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (r={r:.7f}) — dropping: {b}  (kept: {a})')

for a, b, d in near_dup_l2:
    if b not in to_drop and a not in to_drop:
        to_drop.add(b)
        print(f'Near-duplicate (L2={d:.2e}) — dropping: {b}  (kept: {a})')

if not to_drop:
    print('No duplicates detected across all three checks — all spectra are unique.')

dataset_clean = dataset_raw.drop(columns=list(to_drop))

print(f'\nSpectra before deduplication : {dataset_raw.shape[1]}')
print(f'Duplicates removed           : {len(to_drop)}')
print(f'Spectra in final dataset     : {dataset_clean.shape[1]}')

## 8. Preprocessing Pipeline

Same three-step pipeline used throughout this project, applied to every spectrum in the deduplicated dataset:

1. **Savitzky-Golay smoothing** — window = 5 points, polynomial order = 2
2. **ALS baseline correction** — λ = 1×10⁵, p = 0.01, 10 iterations (removes fluorescence background and slow baseline drift)
3. **Non-negative clip** — set any negative residuals to zero
4. **L² normalisation** — scale each spectrum to unit Euclidean norm (removes absolute intensity variation from laser power and exposure-time differences)

In [ ]:
from scipy.signal import savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

def l2_normalize(y):
    n = np.linalg.norm(y)
    return y / n if n > 0 else y

def preprocess(y):
    y = sg_smooth(y)
    y = y - als_baseline(y)
    y = np.clip(y, 0, None)
    return l2_normalize(y)

preprocessed = {col: preprocess(dataset_clean[col].values)
                for col in dataset_clean.columns}

dataset_prep = pd.DataFrame(preprocessed, index=dataset_clean.index)
dataset_prep.index.name = 'Raman Shift'

print('Preprocessing complete.')
print(f'Shape  : {dataset_prep.shape}')
print(f'Any NaN: {dataset_prep.isna().any().any()}')
print()
# Quick sanity: L2 norm of each column should be ~1
norms = np.linalg.norm(dataset_prep.values, axis=0)
print('L2 norms (should all be 1.0):')
for col, n in zip(dataset_prep.columns, norms):
    print(f'  {col:<18} {n:.6f}')

### 8b. Functional Data Representation (B-spline, order 4)

Each preprocessed spectrum is projected onto a cubic B-spline basis (order n = 4, degree = 3) with interior knots placed every 10th wavenumber point. The spline-evaluated spectrum replaces the discrete values, consistent with the functional representation used to build the consensus spectra.

In [ ]:
from scipy.interpolate import make_lsq_spline

wn_arr = dataset_prep.index.values.astype(float)

# Interior knots every 10th point — matches raman_consensus.ipynb
interior_knots = wn_arr[10:-10:10]
k = 3  # degree (order 4 = degree 3, cubic)

spline_fitted = {}
for col in dataset_prep.columns:
    y = dataset_prep[col].values
    spl = make_lsq_spline(wn_arr, y, t=np.concatenate([[wn_arr[0]] * (k + 1),
                                                        interior_knots,
                                                        [wn_arr[-1]] * (k + 1)]),
                          k=k)
    spline_fitted[col] = spl(wn_arr)

dataset_prep = pd.DataFrame(spline_fitted, index=dataset_prep.index)
dataset_prep.index.name = 'Raman Shift'

print(f'B-spline (order 4) fit applied to all {dataset_prep.shape[1]} spectra.')
print(f'Interior knots : {len(interior_knots)}  →  total basis functions ≈ {len(interior_knots) + k + 1}')

# Verify residual (mean max absolute deviation from discrete values)
orig = pd.DataFrame({col: preprocess(dataset_clean[col].values)
                     for col in dataset_clean.columns}, index=dataset_clean.index)
max_dev = np.max(np.abs(dataset_prep.values - orig.values), axis=0)
print(f'Max abs deviation (spline vs discrete): mean={max_dev.mean():.2e}, max={max_dev.max():.2e}')

## 9. Final Dataset Overview (Preprocessed)

In [ ]:
# Rebuild metadata for final columns only
final_cols = list(dataset_clean.columns)
meta_final = meta_df[
    meta_df.apply(lambda r: f"{r['Adulterant']} {r['Concentration']}" in final_cols, axis=1)
].copy()
meta_final['Label'] = meta_final.apply(lambda r: f"{r['Adulterant']} {r['Concentration']}", axis=1)
meta_final = meta_final.set_index('Label')

print('Final dataset — spectrum summary:')
print(f'  Shape   : {dataset_clean.shape[0]} wavenumbers × {dataset_clean.shape[1]} spectra')
print(f'  WN range: {dataset_clean.index.min():.1f} – {dataset_clean.index.max():.1f} cm⁻¹')
print()
print('Adulterant breakdown:')
adult_counts = meta_final['Adulterant'].value_counts()
for adult, cnt in adult_counts.items():
    concs = sorted(meta_final[meta_final['Adulterant'] == adult]['Concentration'].tolist())
    print(f'  {adult:<10}  {cnt} spectra  {concs}')
print()
print('Missing concentration levels per adulterant:')
expected_concs = {'0.5x', '1x', '2x', '3x'}
for adult in sorted(meta_final['Adulterant'].unique()):
    present = set(meta_final[meta_final['Adulterant'] == adult]['Concentration'].tolist())
    missing = expected_concs - present
    corrupt = set()
    # flag corrupt too
    for cf in corrupt_files:
        if cf.startswith(adult + ' '):
            conc = re.search(r'(\d+\.?\d*x)', cf)
            if conc:
                corrupt.add(conc.group(1))
    notes = []
    if missing - corrupt:
        notes.append(f'not collected: {sorted(missing - corrupt)}')
    if corrupt:
        notes.append(f'corrupt file: {sorted(corrupt)}')
    print(f'  {adult:<10}  {"none" if not notes else "; ".join(notes)}')

In [ ]:
# Plot all preprocessed spectra, coloured by adulterant
wn = dataset_prep.index.values

fig, ax = plt.subplots(figsize=(14, 5))
for col in dataset_prep.columns:
    adult = col.split()[0]
    conc  = col.split()[1]
    ax.plot(wn, dataset_prep[col],
            color=ADULTERANT_COLOR.get(adult, 'grey'),
            linewidth=0.9, alpha=0.75,
            marker=CONC_MARKER.get(conc, 'o'), markevery=150, markersize=4)

from matplotlib.patches import Patch
handles = [Patch(color=c, label=a) for a, c in ADULTERANT_COLOR.items()]
ax.legend(handles=handles, fontsize=9, title='Adulterant', loc='upper right')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('All Adulterated Spectra — Preprocessed (SG + ALS + L² norm)')
plt.tight_layout()
plt.show()

In [ ]:
# Per-adulterant panel — preprocessed
adulterants = sorted(meta_final['Adulterant'].unique())
fig, axes   = plt.subplots(len(adulterants), 1, figsize=(14, 4 * len(adulterants)), sharex=True)

conc_palette = {'0.5x': '#1a9850', '1x': '#fee08b', '2x': '#fc8d59', '3x': '#d73027'}

for ax, adult in zip(axes, adulterants):
    cols_a = [c for c in dataset_prep.columns if c.startswith(adult + ' ')]
    for col in cols_a:
        conc = col.split()[1]
        ax.plot(wn, dataset_prep[col],
                color=conc_palette.get(conc, 'grey'),
                linewidth=1.1, label=f'{conc}')
    ax.set_ylabel('Norm. Intensity (a.u.)')
    ax.set_title(f'{adult}  (n={len(cols_a)})')
    ax.legend(title='Concentration', fontsize=8, loc='upper right')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle('Adulterated Spectra by Adulterant — Preprocessed (SG + ALS + L² norm)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

## 10. Intensity Statistics (Preprocessed)

In [ ]:
stats = pd.DataFrame({
    'Adulterant':    [c.split()[0] for c in dataset_prep.columns],
    'Concentration': [c.split()[1] for c in dataset_prep.columns],
    'Min':           dataset_prep.min().values,
    'Max':           dataset_prep.max().values,
    'Mean':          dataset_prep.mean().values,
    'Std':           dataset_prep.std().values,
    'SNR (mean/std)': (dataset_prep.mean() / dataset_prep.std()).values,
}, index=dataset_prep.columns)

print(stats.round(5).to_string())

## 10. Duplicate & Load Report Summary

In [ ]:
print('=' * 70)
print('DATASET CONSTRUCTION REPORT')
print('=' * 70)
print(f'  Source folder       : {SRC_DIR}/')
print(f'  Files discovered    : {len(all_files)}')
print(f'  Corrupt / unreadable: {len(corrupt_files)}')
if corrupt_files:
    for f in corrupt_files:
        print(f'      ✗  {f}')
print(f'  Successfully loaded : {dataset_raw.shape[1]}')
print()
print('  Duplicate checks:')
print(f'    Exact duplicates          : {len(exact_dup_pairs)}')
print(f'    Near-duplicates (r>0.99999): {len(near_dup_corr)}')
print(f'    Near-duplicates (L2<1e-6)  : {len(near_dup_l2)}')
total_removed = len(to_drop)
print(f'    Total removed             : {total_removed}')
print()
print(f'  Final dataset : {dataset_clean.shape[1]} spectra × {dataset_clean.shape[0]} wavenumbers')
print(f'  WN range      : {dataset_clean.index.min():.1f} – {dataset_clean.index.max():.1f} cm⁻¹')
print()
for adult, cnt in adult_counts.items():
    concs = sorted(meta_final[meta_final['Adulterant'] == adult]['Concentration'].tolist())
    print(f'  {adult:<10} : {cnt} spectra a7t {concs}')
print('=' * 70)

## 12. Save Final Dataset

In [ ]:
out_path = 'adulterated_raman_raw.xlsx'

with pd.ExcelWriter(out_path) as writer:
    # Raw intensities (pre-preprocessing)
    dataset_clean.to_excel(writer, sheet_name='Spectra Raw')
    # Preprocessed (SG + ALS + L2 norm)
    dataset_prep.to_excel(writer, sheet_name='Spectra Preprocessed')
    # Metadata for each spectrum
    meta_final.reset_index().rename(columns={'Label': 'Spectrum'}).to_excel(
        writer, sheet_name='Metadata', index=False)
    # Intensity statistics (preprocessed)
    stats.to_excel(writer, sheet_name='Statistics')
    # Load log (all files including corrupt)
    load_report.to_excel(writer, sheet_name='Load Log', index=False)

print(f'Saved → {out_path}')
print(f'  Sheet "Spectra Raw"         : {dataset_clean.shape[1]} spectra × {dataset_clean.shape[0]} wn points (raw counts)')
print(f'  Sheet "Spectra Preprocessed": {dataset_prep.shape[1]} spectra × {dataset_prep.shape[0]} wn points (SG+ALS+L²)')
print(f'  Sheet "Metadata"            : {len(meta_final)} rows')
print(f'  Sheet "Statistics"          : {len(stats)} rows')
print(f'  Sheet "Load Log"            : {len(load_report)} rows')

## 13. Consensus Spectrum vs Adulterated Samples

The Arithmetic Mean consensus spectrum — built from 42 quality-controlled processed milk spectra — is used as the reference standard. Deviations from it highlight spectral features introduced by each adulterant.

**Plot 1:** Consensus vs Urea + Water (chemical/dilution adulterants)  
**Plot 2:** Consensus vs Doxy + Doxy-R + Enro (antibiotic adulterants)

In [ ]:
from scipy.interpolate import interp1d

# Load Arithmetic Mean consensus from raman_consensus_spectra.xlsx
cons_df = pd.read_excel('raman_consensus_spectra.xlsx', index_col=0)
cons_wn = cons_df.index.values.astype(float)
arith_mean = cons_df['Arithmetic Mean'].values

# Regrid consensus onto the adulterated dataset wavenumber axis
wn = dataset_prep.index.values.astype(float)
interp_fn = interp1d(cons_wn, arith_mean, kind='linear',
                     bounds_error=False, fill_value='extrapolate')
arith_mean_reg = interp_fn(wn)

print(f'Consensus WN range : {cons_wn.min():.1f} – {cons_wn.max():.1f} cm⁻¹  ({len(cons_wn)} pts)')
print(f'Adulterated WN range: {wn.min():.1f} – {wn.max():.1f} cm⁻¹  ({len(wn)} pts)')
print(f'Max regrid deviation: {np.abs(cons_wn - wn).max():.4f} cm⁻¹  (regrid applied)')

# ── Plot 1: Consensus vs Urea + Water ────────────────────────────────────────
urea_water_cols = [c for c in dataset_prep.columns if c.startswith('Urea') or c.startswith('Water')]

conc_palette = {'0.5x': '#1a9850', '1x': '#91cf60', '2x': '#fc8d59', '3x': '#d73027'}
adult_style  = {'Urea': '-', 'Water': '--'}

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(wn, arith_mean_reg, color='black', linewidth=2.0,
        label='Consensus (Arithmetic Mean)', zorder=5)

for col in urea_water_cols:
    adult = col.split()[0]
    conc  = col.split()[1]
    ax.plot(wn, dataset_prep[col],
            color=conc_palette.get(conc, 'grey'),
            linestyle=adult_style.get(adult, '-'),
            linewidth=1.2, alpha=0.85,
            label=f'{col}')

ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Arithmetic Mean Consensus vs Urea & Water Adulterated Spectra\n(preprocessed: SG + ALS + L² norm)')
ax.legend(fontsize=8, ncol=2, loc='upper right')
plt.tight_layout()
plt.show()

# ── Plot 2: Consensus vs Antibiotic samples (Doxy, Doxy-R, Enro) ─────────────
antibiotic_cols = [c for c in dataset_prep.columns
                   if c.startswith('Doxy') or c.startswith('Enro')]

antibiotic_color = {'Doxy': '#1f77b4', 'Doxy-R': '#aec7e8', 'Enro': '#ff7f0e'}

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(wn, arith_mean_reg, color='black', linewidth=2.0,
        label='Consensus (Arithmetic Mean)', zorder=5)

for col in antibiotic_cols:
    adult = col.split()[0]
    conc  = col.split()[1]
    ax.plot(wn, dataset_prep[col],
            color=antibiotic_color.get(adult, 'grey'),
            linestyle=adult_style.get(adult, '-'),
            linewidth=1.2, alpha=0.85,
            label=f'{col}')

ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Arithmetic Mean Consensus vs Antibiotic Adulterated Spectra\n(Doxy, Doxy-R, Enro — preprocessed: SG + ALS + L² norm)')
ax.legend(fontsize=8, ncol=2, loc='upper right')
plt.tight_layout()
plt.show()